# catboosting_v2_timeaware.ipynb — 时间感知CV + 稀有合并 + 丢弃流水比率
- **时间感知CV**：以 `issue_time_days` 为时间轴（旧→新）前推式折分，验证更贴近线上。
- **类别稀有合并**：对高基数列（默认 `zip_code`、`title`）做频次阈值合并为 `OTHER`，并加入 `*_freq` 数值特征（仅用训练统计）。
- **丢弃流水比率**：合并流水后删除 `*ratio*` 列（如 `income_expense_ratio`）。
- 仅 CatBoost；保持 v1 参照（若无法解析则用稳健默认）。

In [ ]:

# ==== 配置 ====
REF_DATE_STR = "2025-08-31"
TRAIN_CSV = "train.csv"
TESTAA_CSV = "testaa.csv"
TESTAB_CSV = "testab.csv"

# v2p 流水（若不存在可手改为 v2 或默认名）
TRAIN_STM = "train_statement_feature_v2p.csv"
TESTAA_STM = "testaa_statement_feature_v2p.csv"
TESTAB_STM = "testab_statement_feature_v2p.csv"

OUT_DIR = "outputs_timeaware_v2"
N_FOLDS = 5
RARE_THRESHOLD = 10
RARE_COLS = ['zip_code','title']
DROP_RATIO_COLS = True
USE_GPU = True
RANDOM_STATE = 1337


In [ ]:
# CatBoost 参数
CAT_PARAMS = {
  "iterations": 5000,
  "learning_rate": 0.03,
  "depth": 6,
  "l2_leaf_reg": 9.0,
  "bootstrap_type": "Bernoulli",
  "subsample": 0.75,
  "random_strength": 1.5,
  "one_hot_max_size": 10,
  "loss_function": "Logloss",
  "eval_metric": "AUC",
  "verbose": false
}

In [ ]:

import os, json, numpy as np, pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

os.makedirs(OUT_DIR, exist_ok=True)
REF_DATE = pd.Timestamp(REF_DATE_STR)

def _to_days_since_now(unix_series):
    dt = pd.to_datetime(unix_series, unit='s', utc=True, errors='coerce').dt.tz_convert(None)
    return (REF_DATE - dt).dt.days

def prepare_main_table(df):
    use_cols = [
        'id','title','career','zip_code','residence','loan','term','interest_rate',
        'issue_time','syndicated','installment','record_time','history_time',
        'total_accounts','balance_accounts','balance_limit','balance','level'
    ] + (['label'] if 'label' in df.columns else [])
    df = df[use_cols].copy()
    for c in ['issue_time','record_time','history_time']:
        df[f'{c}_days'] = _to_days_since_now(df[c])
    df['diff_issue_record_days']   = df['issue_time_days'] - df['record_time_days']
    df['diff_issue_history_days']  = df['issue_time_days'] - df['history_time_days']
    df['diff_record_history_days'] = df['record_time_days'] - df['history_time_days']
    df['utilization']    = (df['balance'] / df['balance_limit']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 10)
    df['accounts_ratio'] = (df['balance_accounts'] / df['total_accounts']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 1)
    def split_level(x):
        if isinstance(x, str) and len(x) >= 2: return x[0], x[1:]
        return 'NA', 'NA'
    lv = df['level'].fillna('NA')
    df['grade'], df['subgrade'] = zip(*lv.map(split_level))
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    for c in cat_cols:
        df[c] = df[c].astype(str).fillna('NA')
    return df

def merge_statement_feats(main_df, stm_path):
    if not os.path.exists(stm_path):
        main_df['has_stm'] = 0
        return main_df
    stm = pd.read_csv(stm_path)
    if 'label' in stm.columns: stm = stm.drop(columns=['label'])
    out = main_df.merge(stm, on='id', how='left')
    stm_cols = [c for c in stm.columns if c!='id']
    out['has_stm'] = (out[stm_cols].notna().any(axis=1)).astype(int)
    out[stm_cols] = out[stm_cols].fillna(0)
    if DROP_RATIO_COLS:
        ratio_cols = [c for c in stm_cols if 'ratio' in c.lower()]
        out = out.drop(columns=ratio_cols, errors='ignore')
    return out

def rare_merge_and_add_freq(train_df, test_dfs, cols, thr=10):
    n = len(train_df)
    for c in cols:
        tr_s = train_df[c].astype(str)
        vc = tr_s.value_counts()
        rare_vals = set(vc[vc < thr].index.tolist())
        # 合并稀有
        train_df[c] = tr_s.where(~tr_s.isin(rare_vals), 'OTHER')
        for te in test_dfs:
            if te is None or c not in te.columns: continue
            te_c = te[c].astype(str)
            te[c] = te_c.where(~te_c.isin(rare_vals), 'OTHER')
        # 频次数值特征
        vc2 = train_df[c].value_counts()
        train_df[c + "_freq"] = train_df[c].map(vc2).astype(float).fillna(1.0) / max(n,1)
        for te in test_dfs:
            if te is None or c not in te.columns: continue
            te[c + "_freq"] = te[c].map(vc2).astype(float).fillna(1.0) / max(n,1)
    return train_df, test_dfs

def make_time_series_folds(df, n_folds=5, time_col='issue_time_days'):
    # issue_time_days 越大越旧；这里按 旧->新 的顺序做累积式前推验证
    order = np.argsort(df[time_col].values)[::-1]  # 旧到新
    fold_sizes = np.full(n_folds, len(order)//n_folds, dtype=int)
    fold_sizes[:len(order)%n_folds] += 1
    idx_slices, start = [], 0
    for fs in fold_sizes:
        idx_slices.append(order[start:start+fs])
        start += fs
    folds = []
    for i in range(1, n_folds):
        tr_idx = np.concatenate(idx_slices[:i])
        va_idx = idx_slices[i]
        folds.append((tr_idx, va_idx))
    return folds

def sanitize_cat_params(params, use_gpu=False, y=None, seed=1337):
    p = dict(params)
    if use_gpu and 'rsm' in p:
        p.pop('rsm', None)
    if p.get('bootstrap_type','').lower()=='bayesian' and 'subsample' in p:
        p.pop('subsample', None); p.setdefault('bagging_temperature', 1.0)
    if use_gpu: p['task_type'] = 'GPU'
    if y is not None:
        pos, neg = int(np.sum(y)), int(len(y)-np.sum(y))
        p['scale_pos_weight'] = float(neg/max(pos,1))
    p.setdefault('loss_function','Logloss'); p.setdefault('eval_metric','AUC'); p.setdefault('verbose', False)
    p['random_seed'] = seed
    return p

def train_timeaware_cv(df_train, features, cat_cols, params, n_folds=5, seed=1337):
    X = df_train[features].copy()
    y = df_train['label'].astype(int).values
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]
    p = sanitize_cat_params(params, use_gpu=USE_GPU, y=y, seed=seed)
    folds = make_time_series_folds(df_train, n_folds=n_folds, time_col='issue_time_days')
    oof = np.zeros(len(y), dtype=float)
    fold_scores, models = [], []
    for f,(tr_idx,va_idx) in enumerate(folds, 1):
        train_pool = Pool(X.iloc[tr_idx], label=y[tr_idx], cat_features=cat_idx)
        valid_pool = Pool(X.iloc[va_idx], label=y[va_idx], cat_features=cat_idx)
        model = CatBoostClassifier(**p)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=300)
        pred = model.predict_proba(valid_pool)[:,1]
        oof[va_idx] = pred
        fold_scores.append(roc_auc_score(y[va_idx], pred))
        models.append(model)
    mean_auc = roc_auc_score(y, oof)
    return mean_auc, fold_scores, oof, models


In [ ]:

# ==== 读取与特征合并 ====
tr = pd.read_csv(TRAIN_CSV)
te_aa = pd.read_csv(TESTAA_CSV) if os.path.exists(TESTAA_CSV) else None
te_ab = pd.read_csv(TESTAB_CSV) if os.path.exists(TESTAB_CSV) else None

tr = prepare_main_table(tr)
if te_aa is not None: te_aa = prepare_main_table(te_aa)
if te_ab is not None: te_ab = prepare_main_table(te_ab)

tr = merge_statement_feats(tr, TRAIN_STM)
if te_aa is not None: te_aa = merge_statement_feats(te_aa, TESTAA_STM)
if te_ab is not None: te_ab = merge_statement_feats(te_ab, TESTAB_STM)

# 稀有合并 + 频次（仅用训练统计，避免泄漏）
tr, [te_aa, te_ab] = rare_merge_and_add_freq(tr, [te_aa, te_ab], cols=RARE_COLS, thr=RARE_THRESHOLD)

drop_cols = ['id','label']
features = [c for c in tr.columns if c not in drop_cols]
cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
cat_cols = [c for c in cat_cols if c in features]

print("Train shape:", tr.shape, "| #features:", len(features), "| #cat:", len(cat_cols))


In [ ]:

# ==== 时间感知 CV 训练 ====
mean_auc, fold_scores, oof, models = train_timeaware_cv(tr, features, cat_cols, CAT_PARAMS, n_folds=N_FOLDS, seed=RANDOM_STATE)
pd.DataFrame({'id': tr['id'], 'label': tr['label'], 'oof_pred': oof}).to_csv(os.path.join(OUT_DIR, "oof_timeaware.csv"), index=False, encoding='utf-8')
print("[TimeAware CV] OOF AUC =", round(float(mean_auc), 6), "| folds:", [round(float(x),6) for x in fold_scores])


In [ ]:

# ==== 测试预测 ====
def predict_one(df, tag):
    if df is None:
        print(f"[INFO] 无 {tag} 测试集，跳过。"); return
    X = df[features].copy()
    pool = Pool(X, cat_features=[X.columns.get_loc(c) for c in cat_cols])
    preds = np.mean([m.predict_proba(pool)[:,1] for m in models], axis=0)
    out = pd.DataFrame({'id': df['id'], 'prob': preds})
    path = os.path.join(OUT_DIR, f"test_{tag}_pred_timeaware.csv")
    out.to_csv(path, index=False, encoding='utf-8')
    print("[SAVE]", path)

predict_one(te_aa, "aa")
predict_one(te_ab, "ab")
